In [5]:
import pandas as pd
import os
from datetime import datetime
import string
import random
import itertools
import uuid
from typing import List, Dict, Optional
import psutil
import json

def generate_matrix_code() -> str:
    """Generate a matrix code starting with 'SCO' followed by 4 random uppercase letters."""
    return "SCO" + ''.join(random.choice(string.ascii_uppercase) for _ in range(4))

def px_escape(text: str) -> str:
    """Escape quotes in strings for PX format."""
    if isinstance(text, str):
        return text.replace('"', '""')
    return str(text)

def load_metadata(metadata_file: Optional[str]) -> Optional[Dict]:
    """Load metadata from JSON file if provided."""
    if metadata_file and os.path.exists(metadata_file):
        try:
            with open(metadata_file, 'r', encoding='utf-8') as f:
                return json.load(f)
        except UnicodeDecodeError:
            print("UTF-8 decoding failed for metadata. Trying 'latin-1'...")
            with open(metadata_file, 'r', encoding='latin-1') as f:
                return json.load(f)
    return None

def extract_metadata_info(metadata: Dict) -> Dict:
    """Extract relevant metadata fields and infer dimensions."""
    if not metadata or not isinstance(metadata, list) or not metadata:
        return {}

    meta = metadata[0]
    info = {
        'title': meta.get('http://purl.org/dc/terms/title', [{}])[0].get('@value', 'Untitled Dataset'),
        'subject_area': meta.get('http://www.w3.org/ns/dcat#theme', [{}])[0].get('@id', 'General').split('/')[-1],
        'description': meta.get('http://purl.org/dc/terms/description', [{}])[0].get('@value', ''),
        'source': meta.get('http://purl.org/dc/terms/publisher', [{}])[0].get('@id', 'Unknown').split('/')[-1],
        'units': meta.get('http://statistics.gov.scot/def/statistical-quality/relevance', [{}])[0].get('@value', 'Unknown')
    }

    structure = meta.get('http://purl.org/linked-data/cube#structure', [{}])[0].get('@id', '')
    graph = meta.get('http://publishmydata.com/def/dataset#graph', [{}])[0].get('@id', '')
    
    potential_dims = []
    for uri in [structure, graph]:
        if uri:
            dim = uri.split('/')[-1].replace('-', ' ').title()
            potential_dims.append(dim)

    themes = [t.get('@id', '').split('/')[-1].replace('-', ' ').title() 
              for t in meta.get('http://www.w3.org/ns/dcat#theme', []) if t.get('@id')]
    potential_dims.extend(themes)

    potential_dims = list(dict.fromkeys([d for d in potential_dims if d]))
    info['potential_dimensions'] = potential_dims
    return info

def define_dimensions(df: pd.DataFrame, dimension_cols: List[str], max_values_per_dim: Optional[int] = None) -> Dict[str, Dict[str, List[str]]]:
    """Define dimension values and codes dynamically from DataFrame."""
    dimensions = {}
    for col in dimension_cols:
        # Get unique values, excluding nulls and invalid entries
        valid_values = df[col].dropna().astype(str)
        valid_values = valid_values[~valid_values.isin(['<NA>', 'nan', ''])]
        
        unique_vals = sorted(valid_values.unique())
        
        # Limit values if max_values_per_dim is set
        if max_values_per_dim and len(unique_vals) > max_values_per_dim:
            print(f"Warning: Column '{col}' has {len(unique_vals)} unique values. Limiting to top {max_values_per_dim}.")
            value_counts = df[col].value_counts()
            top_values = value_counts.nlargest(max_values_per_dim).index.tolist()
            unique_vals = sorted(top_values)
            df.loc[~df[col].isin(top_values), col] = "Other"
        
        if not unique_vals:
            raise ValueError(f"No valid unique values for {col} after cleaning.")
        
        dimensions[col] = {
            "values": unique_vals,
            "codes": [f"{i+1:02d}" for i in range(len(unique_vals))]
        }
    return dimensions

def infer_dimensions_from_metadata_and_data(df: pd.DataFrame, metadata_info: Dict) -> tuple[List[str], List[str]]:
    """Infer stub and heading columns from metadata and DataFrame columns."""
    potential_dims = metadata_info.get('potential_dimensions', [])
    df_cols = list(df.columns)

    stub_cols = []
    heading_cols = []

    for dim in potential_dims:
        norm_dim = dim.lower().replace(' ', '')
        for col in df_cols:
            norm_col = col.lower().replace(' ', '')
            if norm_dim in norm_col or norm_col in norm_dim:
                if not stub_cols and not heading_cols:
                    stub_cols.append(col)
                elif col not in stub_cols:
                    heading_cols.append(col)

    if not stub_cols and not heading_cols:
        stub_cols = [df_cols[0]] if df_cols else []
        heading_cols = [col for col in df_cols[1:] if col != 'Count']

    return stub_cols, heading_cols

def px_values_and_codes(name: str, dim: Dict) -> str:
    """Generate VALUES and CODES blocks, ensuring valid lengths."""
    max_val_length = 256
    values = dim["values"]
    cleaned_values = []
    for v in values:
        str_v = str(v)
        if len(str_v) > max_val_length:
            print(f"Warning: Value '{str_v}' in '{name}' exceeds {max_val_length} chars. Truncating.")
            str_v = str_v[:max_val_length]
        if len(str_v) == 0:
            print(f"Warning: Empty value in '{name}'. Replacing with 'Unknown'.")
            str_v = "Unknown"
        cleaned_values.append(str_v)
    quoted_vals = ",".join(f'"{px_escape(v)}"' for v in cleaned_values)
    quoted_codes = ",".join(f'"{px_escape(str(c))}"' for c in dim["codes"])
    return f'VALUES("{name}")={quoted_vals};\nCODES("{name}")={quoted_codes};\n'

def tidy_to_pxstat(
    input_file: str,
    output_file: Optional[str] = None,
    metadata_file: Optional[str] = None,
    stub_cols: Optional[List[str]] = None,
    heading_cols: Optional[List[str]] = None,
    value_col: str = "Count",
    decimals: Optional[int] = 0,
    agg_method: str = "sum",
    chunk_size: int = 10000,
    max_combinations: Optional[int] = None,
    max_values_per_dim: Optional[int] = None
) -> str:
    """
    Convert Tidy format CSV to monolingual PxStat format with dynamic dimensions.
    """
    print(f"Loading data from {input_file}...")
    try:
        # Load input file
        if not input_file.lower().endswith('.csv'):
            raise ValueError("Input file must be CSV")
        try:
            df = pd.read_csv(input_file, low_memory=False, dtype_backend="numpy_nullable", encoding='utf-8')
        except UnicodeDecodeError:
            print("UTF-8 decoding failed. Trying 'latin-1' encoding...")
            df = pd.read_csv(input_file, low_memory=False, dtype_backend="numpy_nullable", encoding='latin-1')
        except UnicodeDecodeError:
            print("Latin-1 decoding failed. Trying 'windows-1252' encoding...")
            df = pd.read_csv(input_file, low_memory=False, dtype_backend="numpy_nullable", encoding='windows-1252')

        print(f"Data loaded with {len(df)} rows and {len(df.columns)} columns")
        print(f"Columns: {', '.join(df.columns)}")

        # Load metadata
        metadata = load_metadata(metadata_file)
        meta_info = extract_metadata_info(metadata) if metadata else {}

        # Validate required columns
        if value_col not in df.columns:
            raise ValueError(f"Value column '{value_col}' not found in input data")

        # Infer stub and heading columns if not provided
        if not stub_cols or not heading_cols:
            inferred_stub_cols, inferred_heading_cols = infer_dimensions_from_metadata_and_data(df, meta_info)
            stub_cols = stub_cols or inferred_stub_cols
            heading_cols = heading_cols or inferred_heading_cols

        group_cols = [col for col in stub_cols + heading_cols if col != value_col]

        if not group_cols:
            raise ValueError("No dimension columns specified or inferred")

        # Validate dimension columns
        missing_cols = [col for col in group_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing dimension columns: {missing_cols}")

        # Convert categorical columns to string to handle mixed types
        for col in group_cols:
            df[col] = df[col].astype(str).replace(['<NA>', 'nan', ''], 'Unknown')

        # Simplify to needed columns
        df_simple = df[group_cols + [value_col]].copy()

        # Make values numeric
        df_simple[value_col] = pd.to_numeric(df_simple[value_col], errors="coerce")
        if df_simple[value_col].isna().any():
            print(f"Warning: {df_simple[value_col].isna().sum()} non-numeric values in {value_col} set to NaN.")

        # Aggregate duplicates
        print(f"Aggregating data with {agg_method}...")
        df_agg = df_simple.groupby(group_cols, as_index=False).agg({value_col: agg_method})
        print(f"Aggregated to {len(df_agg)} unique combinations")

        # Define dimensions dynamically
        dimensions = define_dimensions(df_simple, group_cols, max_values_per_dim)

        # Calculate expected combinations
        dim_counts = [len(dimensions[col]["values"]) for col in group_cols]
        expected_count = 1
        for count in dim_counts:
            expected_count *= count
        print(f"Dimension value counts: {list(zip(group_cols, dim_counts))}")
        print(f"Expected data points: {expected_count}")

        if max_combinations and expected_count > max_combinations:
            raise ValueError(f"Too many combinations ({expected_count}). Reduce dimensions or unique values, or increase max_combinations.")

        # Log memory usage
        process = psutil.Process()
        mem_info = process.memory_info()
        print(f"Current memory usage: {mem_info.rss / 1024**2:.2f} MB")

        # Create a mapping from aggregated data
        print("Creating data mapping...")
        data_map = {
            tuple(row[col] for col in group_cols): row[value_col]
            for _, row in df_agg.iterrows()
        }

        # Metadata for PxStat file
        creation_date = datetime.now().strftime("%Y%m%d %H:%M")
        title = meta_info.get('title', f"Dataset from {os.path.basename(input_file)}")
        subject_area = meta_info.get('subject_area', 'General')
        matrix_code = generate_matrix_code()
        units = px_escape(meta_info.get('units', 'Count'))
        source = meta_info.get('source', 'Unknown')

        header = f"""CHARSET="UTF-16";
AXIS-VERSION="2013";
CREATION-DATE="{creation_date}";
MATRIX="{matrix_code}";
DECIMALS={decimals};
SUBJECT-AREA="{px_escape(subject_area)}";
SUBJECT-CODE="{matrix_code[:4] if len(matrix_code) >= 4 else matrix_code}";
CONTENTS="{px_escape(title)}";
TITLE="{px_escape(title)} - by {', '.join(px_escape(col) for col in group_cols)}";
UNITS="{units}";
STUB="{','.join(f'"{px_escape(col)}"' for col in stub_cols)}";
HEADING="{','.join(f'"{px_escape(col)}"' for col in heading_cols)}";
SOURCE="{px_escape(source)}";
"""

        # Generate VALUES and CODES blocks
        meta_parts = "".join(px_values_and_codes(col, dimensions[col]) for col in group_cols)

        # Output filename
        output_file = output_file or os.path.splitext(input_file)[0] + ".px"

        # Write to file with chunked processing
        print(f"Writing {expected_count} data points to {output_file}...")
        with open(output_file, "w", encoding="utf-16") as f:
            print("Writing header...")
            f.write(header)
            print("Writing metadata parts...")
            f.write(meta_parts)
            print("Writing DATA section...")
            f.write("DATA=\n")

            # Process combinations in chunks
            dim_values = [dimensions[col]["values"] for col in group_cols]
            print(f"Dimension values counts: {[(col, len(dimensions[col]['values'])) for col in group_cols]}")
            product_iterator = itertools.product(*dim_values)
            data_values_written = 0

            try:
                while True:
                    chunk = list(itertools.islice(product_iterator, chunk_size))
                    if not chunk:
                        print("No more chunks to process.")
                        break
                    print(f"Processing chunk of {len(chunk)} combinations...")
                    data_values = []
                    for combo in chunk:
                        value = data_map.get(combo, pd.NA)
                        data_values.append(".." if pd.isna(value) else str(int(value) if decimals == 0 else round(value, decimals)))
                    f.write(" ".join(data_values) + "\n")
                    data_values_written += len(data_values)
                    mem_info = process.memory_info()
                    print(f"Processed {data_values_written}/{expected_count} data points (Memory: {mem_info.rss / 1024**2:.2f} MB)")
            except Exception as e:
                print(f"Error in chunked processing: {str(e)}")
                raise

            print("Writing final semicolon...")
            f.write(";")

        # Verify data count
        if data_values_written != expected_count:
            raise ValueError(f"Data count mismatch: expected {expected_count}, wrote {data_values_written}")

        print(f"✅ PX file saved as: {output_file}")
        return output_file

    except Exception as e:
        print(f"❌ Error processing file: {str(e)}")
        raise

if __name__ == "__main__":
    # Preprocess CSV
    input_file = "Data Zone Lookup - Archived Geographies.csv"
    try:
        df = pd.read_csv(input_file, encoding='utf-8')
    except UnicodeDecodeError:
        print("UTF-8 decoding failed. Using 'latin-1' encoding...")
        df = pd.read_csv(input_file, encoding='latin-1')

    # Append a single row with a specific date
    new_row = {
        'DZ2011_Code': 'S01099999',
        'DZ2011_Name': 'Placeholder Zone',
        'Date_from': '2025-05-14',
        'Date_to': '2025-05-14',
        'Geography_Type': 'Scottish Parliamentary Constituency',
        'Geography_Code': 'S16000999',
        'Geography_Name': 'Placeholder Constituency',
        'Count': 1
    }
    new_row_df = pd.DataFrame([new_row])
    df = pd.concat([df, new_row_df], ignore_index=True)

    # Create Year from Date_to
    df['Year'] = pd.to_datetime(df['Date_to'], errors='coerce').dt.year.astype(str).replace('nan', 'Unknown')
    df['Count'] = df['Count'].fillna(1).astype(int)

    # Save with UTF-8 encoding
    df.to_csv(input_file, encoding='utf-8', index=False)

    # Example usage for CSV conversion
    CONFIG = {
        "input_file": "Data Zone Lookup - Archived Geographies.csv",
        "output_file": "Data Zone Lookup - Archived Geographies.px",
        "metadata_file": "DZ11_lookup_archived_geogs_metadata.json",
        "value_col": "Count",
        "stub_cols": ["DZ2011_Code", "Geography_Code"],
        "heading_cols": ["Geography_Type", "Year"],
        "decimals": 0,
        "agg_method": "sum",
        "chunk_size": 10000,
        "max_combinations": 2000000,
        "max_values_per_dim": 100
    }

    # Run conversion
    print("Running conversion with the following configuration:")
    for key, value in CONFIG.items():
        print(f"{key}: {value}")
    tidy_to_pxstat(**CONFIG)

Running conversion with the following configuration:
input_file: Data Zone Lookup - Archived Geographies.csv
output_file: Data Zone Lookup - Archived Geographies.px
metadata_file: DZ11_lookup_archived_geogs_metadata.json
value_col: Count
stub_cols: ['DZ2011_Code', 'Geography_Code']
heading_cols: ['Geography_Type', 'Year']
decimals: 0
agg_method: sum
chunk_size: 10000
max_combinations: 2000000
max_values_per_dim: 100
Loading data from Data Zone Lookup - Archived Geographies.csv...
Data loaded with 111617 rows and 9 columns
Columns: DZ2011_Code, DZ2011_Name, Date_from, Date_to, Geography_Type, Geography_Code, Geography_Name, Count, Year
Aggregating data with sum...
Aggregated to 86179 unique combinations
Dimension value counts: [('DZ2011_Code', 100), ('Geography_Code', 100), ('Geography_Type', 4), ('Year', 6)]
Expected data points: 240000
Current memory usage: 229.27 MB
Creating data mapping...
Writing 240000 data points to Data Zone Lookup - Archived Geographies.px...
Writing header...
